<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания № 17


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать  базовый  класс ShippingOption в  C#,  который  будет  представлять 
различные опции доставки. На основе этого класса разработать 2-3 производных 
класса, демонстрирующих принципы наследования и полиморфизма. В каждом из 
классов  должны  быть  реализованы  новые  атрибуты  и  методы,  а  также 
переопределены  некоторые  методы  базового  класса  для  демонстрации 
полиморфизма.

#### Дополнительное задание
Добавьте к сущестующим классам конструктора классов с использованием гетторов и сетторов и реализуйте взаимодействие объектов между собой

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [3]:
// Базовый класс ShippingOption
public class ShippingOption
{
    private string _deliveryOptionId;
    private string _deliveryOptionName;
    private decimal _cost;

    public string DeliveryOptionId 
    { 
        get => _deliveryOptionId;
        set => _deliveryOptionId = !string.IsNullOrEmpty(value) ? value : throw new ArgumentException("ID не может быть пустым");
    }
    
    public string DeliveryOptionName 
    { 
        get => _deliveryOptionName;
        set => _deliveryOptionName = !string.IsNullOrEmpty(value) ? value : throw new ArgumentException("Название не может быть пустым");
    }
    
    public decimal Cost 
    { 
        get => _cost;
        set => _cost = value >= 0 ? value : throw new ArgumentException("Стоимость не может быть отрицательной");
    }

    public ShippingOption(string id, string name, decimal cost)
    {
        DeliveryOptionId = id;
        DeliveryOptionName = name;
        Cost = cost;
    }

    public virtual decimal CalculateCost() => Cost;
    
    public virtual string EstimateDeliveryTime() => "Время доставки не указано";
    
    public virtual string GetDeliveryDetails() => 
        $"ID: {DeliveryOptionId}, Название: {DeliveryOptionName}, Стоимость: {Cost} руб.";

    // Метод для взаимодействия с другими опциями доставки
    public virtual bool CanCombineWith(ShippingOption other) => false;
}

// Стандартная доставка
public class StandardDelivery : ShippingOption
{
    private int _averageDeliveryTime;

    public int AverageDeliveryTime 
    { 
        get => _averageDeliveryTime;
        set => _averageDeliveryTime = value > 0 ? value : throw new ArgumentException("Время доставки должно быть положительным");
    }

    public StandardDelivery(string id, string name, decimal cost, int avgTime) 
        : base(id, name, cost)
    {
        AverageDeliveryTime = avgTime;
    }

    public override string EstimateDeliveryTime() => 
        $"Среднее время: {AverageDeliveryTime} дней";

    public override string GetDeliveryDetails() => 
        base.GetDeliveryDetails() + $", Тип: Стандартная, {EstimateDeliveryTime()}";

    // Может комбинироваться с самовывозом
    public override bool CanCombineWith(ShippingOption other) => 
        other is Pickup;
}

// Экспресс доставка
public class ExpressDelivery : ShippingOption
{
    private int _minDeliveryTime;

    public int MinDeliveryTime 
    { 
        get => _minDeliveryTime;
        set => _minDeliveryTime = value > 0 ? value : throw new ArgumentException("Время доставки должно быть положительным");
    }

    public ExpressDelivery(string id, string name, decimal cost, int minTime) 
        : base(id, name, cost)
    {
        MinDeliveryTime = minTime;
    }

    public override decimal CalculateCost() => Cost * 1.5m;

    public override string EstimateDeliveryTime() => 
        $"Минимальное время: {MinDeliveryTime} дней";

    public override string GetDeliveryDetails() => 
        base.GetDeliveryDetails() + $", Тип: Экспресс, {EstimateDeliveryTime()}, Итог: {CalculateCost()} руб.";

    // Не может комбинироваться с другими экспресс доставками
    public override bool CanCombineWith(ShippingOption other) => 
        !(other is ExpressDelivery);
}

// Самовывоз
public class Pickup : ShippingOption
{
    private string _pickupAddress;

    public string PickupAddress 
    { 
        get => _pickupAddress;
        set => _pickupAddress = !string.IsNullOrEmpty(value) ? value : throw new ArgumentException("Адрес не может быть пустым");
    }

    public Pickup(string id, string name, string address) 
        : base(id, name, 0)
    {
        PickupAddress = address;
    }

    public override string GetDeliveryDetails() => 
        base.GetDeliveryDetails() + $", Тип: Самовывоз, Адрес: {PickupAddress}";

    public override string EstimateDeliveryTime() => "Готов к выдаче через 2 часа";

    // Может комбинироваться со стандартной доставкой
    public override bool CanCombineWith(ShippingOption other) => 
        other is StandardDelivery;
}

// Класс для управления заказами и взаимодействия опций доставки
public class OrderManager
{
    private List<ShippingOption> _availableOptions;

    public OrderManager()
    {
        _availableOptions = new List<ShippingOption>();
    }

    public void AddShippingOption(ShippingOption option)
    {
        _availableOptions.Add(option);
        Console.WriteLine($"Добавлена опция: {option.DeliveryOptionName}");
    }

    public void ShowAllOptions()
    {
        Console.WriteLine("\n=== Все доступные опции доставки ===");
        foreach (var option in _availableOptions)
        {
            Console.WriteLine(option.GetDeliveryDetails());
        }
    }

    public void CheckCombinations(ShippingOption option1, ShippingOption option2)
    {
        bool canCombine1 = option1.CanCombineWith(option2);
        bool canCombine2 = option2.CanCombineWith(option1);
        
        Console.WriteLine($"\nПроверка комбинации {option1.DeliveryOptionName} + {option2.DeliveryOptionName}:");
        Console.WriteLine($"{option1.DeliveryOptionName} может комбинироваться с {option2.DeliveryOptionName}: {canCombine1}");
        Console.WriteLine($"{option2.DeliveryOptionName} может комбинироваться с {option1.DeliveryOptionName}: {canCombine2}");
        
        if (canCombine1 && canCombine2)
        {
            Console.WriteLine("Опции могут быть использованы вместе!");
        }
        else
        {
            Console.WriteLine("Опции не могут быть использованы вместе");
        }
    }

    public decimal CalculateTotalCost(params ShippingOption[] options)
    {
        decimal total = 0;
        foreach (var option in options)
        {
            total += option.CalculateCost();
        }
        return total;
    }
}

// Глобальный код (точка входа)
var manager = new OrderManager();

// Создаем опции доставки
var standard = new StandardDelivery("1", "Стандарт", 300, 5);
var express = new ExpressDelivery("2", "Экспресс", 300, 2);
var pickup = new Pickup("3", "Самовывоз", "ул. Центральная, 15");

// Добавляем опции в менеджер
manager.AddShippingOption(standard);
manager.AddShippingOption(express);
manager.AddShippingOption(pickup);

// Показываем все опции
manager.ShowAllOptions();

// Проверяем взаимодействие между опциями
Console.WriteLine("\n=== Проверка взаимодействия опций ===");
manager.CheckCombinations(standard, pickup);
manager.CheckCombinations(express, standard);
manager.CheckCombinations(express, express);

// Расчет общей стоимости
Console.WriteLine($"\n=== Расчет стоимости ===");
Console.WriteLine($"Общая стоимость стандарт + экспресс: {manager.CalculateTotalCost(standard, express)} руб.");
Console.WriteLine($"Общая стоимость стандарт + самовывоз: {manager.CalculateTotalCost(standard, pickup)} руб.");

// Демонстрация работы геттеров/сеттеров с валидацией
try
{
    var invalidOption = new StandardDelivery("", "Невалидная", -100, -5);
}
catch (ArgumentException ex)
{
    Console.WriteLine($"\nОшибка валидации: {ex.Message}");
}

Добавлена опция: Стандарт
Добавлена опция: Экспресс
Добавлена опция: Самовывоз

=== Все доступные опции доставки ===
ID: 1, Название: Стандарт, Стоимость: 300 руб., Тип: Стандартная, Среднее время: 5 дней
ID: 2, Название: Экспресс, Стоимость: 300 руб., Тип: Экспресс, Минимальное время: 2 дней, Итог: 450.0 руб.
ID: 3, Название: Самовывоз, Стоимость: 0 руб., Тип: Самовывоз, Адрес: ул. Центральная, 15

=== Проверка взаимодействия опций ===

Проверка комбинации Стандарт + Самовывоз:
Стандарт может комбинироваться с Самовывоз: True
Самовывоз может комбинироваться с Стандарт: True
Опции могут быть использованы вместе!

Проверка комбинации Экспресс + Стандарт:
Экспресс может комбинироваться с Стандарт: True
Стандарт может комбинироваться с Экспресс: False
Опции не могут быть использованы вместе

Проверка комбинации Экспресс + Экспресс:
Экспресс может комбинироваться с Экспресс: False
Экспресс может комбинироваться с Экспресс: False
Опции не могут быть использованы вместе

=== Расчет стоимости